In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
import joblib 
import numpy as np

# from sklearn.preprocessing import PolynomialFeatures
# from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, ElasticNet

# 1. Tải tập dữ liệu từ quá trình Trace-driven Simulation
df = pd.read_csv('../trace_driven_simulation/data/simulation_data_198.csv')
TAU_SLO = 200
df_clean = df[df['response_time'] <= TAU_SLO].copy()
df_clean['slo_threshold'] = TAU_SLO

# Features: requests, SLO_threshold | Label: vms_count
X = df_clean[['requests', 'slo_threshold']]
y = df_clean['vms_count']

In [2]:
# 2. Chia dữ liệu (Random shuffle để bao phủ dải tải)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [3]:
X_train

,requests,slo_threshold
151,35530,200
89,21270,200
34,8620,200
169,39670,200
187,43810,200
...,...,...
91,21730,200
136,32080,200
18,4940,200
118,27940,200


In [4]:
# 3. Định nghĩa các mô hình
model_M = DecisionTreeRegressor()
# model_M = ElasticNet()
# model_M = LinearRegression()

In [5]:
# 4. Huấn luyện và đánh giá
model_M.fit(X_train, y_train)
y_pred = model_M.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
print(f"Root Mean Squared Error (RMSE) : {rmse:.4f}")


# # Kiểm tra tham số của mô hình
# print(model_M.get_params()['criterion'])

Root Mean Squared Error (RMSE) : 0.1796


In [6]:
# --- TẠO BẢNG SO SÁNH THỰC TẾ VÀ DỰ ĐOÁN ---
comparison_df = pd.DataFrame({
    'Thứ tự (Phút)': df_clean.loc[X_test.index, 'minute'].values,  # Lấy chính xác phút từ file gốc
    'Số requests': X_test['requests'].values,
    'Thực tế (vm_count)': y_test,
    # Làm tròn kết quả dự đoán thành số nguyên vì số lượng VM không thể là số lẻ
    'Dự đoán (predicted_vm)': y_pred.round().astype(int) 
})
# In ra 15 dòng đầu tiên để kiểm tra trực quan
print("BẢNG SO SÁNH SỐ LƯỢNG MÁY CHỦ (THỰC TẾ vs DỰ ĐOÁN):")
print(comparison_df.head(10))

BẢNG SO SÁNH SỐ LƯỢNG MÁY CHỦ (THỰC TẾ vs DỰ ĐOÁN):
     Thứ tự (Phút)  Số requests  Thực tế (vm_count)  Dự đoán (predicted_vm)
19              20         5170                   5                       5
121            122        28630                  28                      28
195            196        45650                  44                      44
135            136        31850                  31                      31
140            141        33000                  32                      32
84              85        20120                  20                      19
23              24         6090                   6                       6
58              59        14140                  14                      14
46              47        11380                  11                      11
71              72        17130                  17                      17


In [7]:
# 7. Lưu trữ mô hình để dùng cho Algorithm 2
model_filename = './model/predictive_autoscaling_model_M_1.pkl'
joblib.dump(model_M, model_filename)
print(f"Đã lưu mô hình thành công tại: {model_filename}")

Đã lưu mô hình thành công tại: ./model/predictive_autoscaling_model_M_1.pkl


In [8]:
y_pred

array([ 5., 28., 44., 31., 32., 19.,  6., 14., 11., 17., 25., 41.,  7.,
       43., 25., 27., 35.,  5.,  4., 33.,  9., 13., 39., 20., 40., 10.,
       28., 10., 23., 20., 41.])